# Rede de pré-requisitos — Engenharia da Computação (UFRN)

**Disciplina:** Algoritmos e Estruturas de Dados II · **Autor:** Israel Soares de Castro Filho

**Roteiro deste notebook**

| # | Seção | O que você encontra |
|---|---|---|
| 1 | Problema e Pergunta | o problema real e a pergunta que a rede ajuda a responder |
| 2 | Dados utilizados | de onde vêm os dados e como foram validados |
| 3 | Modelagem dos dados em um grafo | o que é nó, aresta, peso, direção e o que ficou de fora |
| 4 | Métodos utilizados para análise | cada método: o que é, de que **semana** do curso vem e o código que o aplica |
| 5 | Como a análise é interpretada | como ler cada número e quais são os cuidados |
| 6 | Resultados | a resposta à pergunta, os limites e a tabela de conceitos por semana |

**Para rodar:** `pip install -r requirements.txt` e depois *Run All*. Os CSVs devem estar em `dados/raw/`.

---
# 1. Problema e Pergunta

## 1.1 O problema

Um curso de graduação é organizado em **períodos**, e muitas disciplinas só podem ser cursadas depois de outras (os **pré-requisitos**). Isso cria uma teia de dependências: reprovar ou adiar uma disciplina pode travar várias outras e, no pior caso, **atrasar a formatura**.

Esse é um problema real para qualquer estudante: *"em quais disciplinas eu não posso me dar ao luxo de errar?"* E também para a coordenação: *"quais disciplinas merecem mais atenção, reforço ou oferta extra?"*

Como as dependências são relações entre pares (A é pré-requisito de B), o currículo é naturalmente uma **rede**.

## 1.2 A pergunta

> **Quais disciplinas obrigatórias de Engenharia da Computação (UFRN) são os principais pontos críticos do currículo — as que mais bloqueiam outras, ligam o início ao fim do curso ou concentram conexões?**

"Ponto crítico" tem vários sentidos possíveis, então a pergunta é desdobrada em quatro perguntas menores, cada uma respondida por um tipo de medida da rede:

| Definições de Crítico | Pergunta | Medidas usadas |
|---|---|---|
| **Gargalo imediato** | quem tem mais disciplinas dependendo diretamente dela? | grau de saída |
| **Efeito cascata** | quanto do curso trava se ela não for cumprida? | alcance (descendentes), horas bloqueadas |
| **Ponte** | quem fica no meio dos caminhos entre o básico e o avançado? | betweenness, closeness |
| **Núcleo denso** | quais formam o bloco mais interligado do currículo? | k-core, k-shell, coreness |

Além disso, uma pergunta de planejamento: **qual é a maior cadeia de pré-requisitos** (o que impõe um mínimo de períodos para se formar)?

**Por que uma rede ajuda?** Olhar uma disciplina isolada mostra só seus pré-requisitos diretos. A estrutura da rede mostra o que está *além*: dependências de dependências, caminhos que passam por uma disciplina e blocos densos. Uma lista de disciplinas não mostra isso.

---
# 2. Dados utilizados

## 2.1 Fonte e arquivos

Foram utilizados os dados reais disponíveis publicamente em `sigaa.ufrn.br`, nesse projeto foram utilizados os dados do currículo de Engenharia da Computação da UFRN (disciplinas, carga horária, período e seus pré-requisitos)

| Arquivo (`dados/raw/`) | Uma linha é… | Colunas principais |
|---|---|---|
| `disciplinas.csv` | uma disciplina (170 linhas: obrigatórias, optativas e complementares) | `codigo`, `nome`, `ch` (carga horária), `natureza`, `nivel` (período), e as colunas brutas de pré-requisitos, co-requisitos e equivalências |
| `arestas.csv` | um pré-requisito (87 linhas) | `pre_requisito` → `disciplina`, e `alternativa` (1 se a disciplina aceita também uma alternativa **fora da grade atual**, indicada por um **"OU"** no texto de pré-requisitos) |

As arestas foram extraídas do texto bruto de pré-requisitos de cada disciplina. A coluna `alternativa` vale 1 quando o pré-requisito vem acompanhado de um "OU" com uma opção **fora da grade atual**: código de um currículo antigo ou disciplina equivalente de outro curso. Exemplo: Eletrônica (`DCA3506`) tem pré-requisito `ECT3511 OU DCA0105`. A `DCA0105` não é mais ofertada, mas continua aceita para quem a cursou antes da mudança de currículo. Só `ECT3511`, que pertence à grade, vira aresta do grafo.

## 2.2 Leitura e validação

Antes de modelar, conferimos se os dados fazem sentido: códigos repetidos, arestas repetidas, laços (A → A), códigos que não existem em `disciplinas.csv` e se a coluna `alternativa` bate com o texto bruto.

In [ ]:
# ------------------------------------------------------------------
# Inicialização de bibliotecas
# ------------------------------------------------------------------

import numpy as np
import pandas as pd
import networkx as nx
import plotly
import plotly.graph_objects as go
import plotly.io as pio
import nbformat
from plotly.subplots import make_subplots
from IPython.display import display

print(f"networkx {nx.__version__} | plotly {plotly.__version__} | pandas {pd.__version__}")

In [ ]:
# ------------------------------------------------------------------
# Caminho de Arquivos
# ------------------------------------------------------------------

RAW_PATH = "dados/raw"
OUTPUT_PATH = "dados/processed"

ARQ_DISCIPLINAS = f"{RAW_PATH}/disciplinas.csv"
ARQ_ARESTAS = f"{RAW_PATH}/arestas.csv"
ARQ_SAIDA = f"{OUTPUT_PATH}/metricas_disciplinas.csv"

In [ ]:
disc = pd.read_csv(ARQ_DISCIPLINAS)
arest = pd.read_csv(ARQ_ARESTAS)

for df_ in (disc, arest):
    for c in df_.columns:
        if not pd.api.types.is_numeric_dtype(df_[c]):
            df_[c] = df_[c].astype("string").str.strip()

print(f"disciplinas.csv: {len(disc)} linhas | arestas.csv: {len(arest)} linhas")
display(disc["natureza"].value_counts().rename("disciplinas").to_frame())

# --- consistência ---

codigos_disc = set(disc["codigo"])
print("códigos repetidos em disciplinas.csv  :", int(disc["codigo"].duplicated().sum()))
print("arestas repetidas                     :", int(arest.duplicated(["pre_requisito", "disciplina"]).sum()))
print("laços (A -> A)                        :", int((arest["pre_requisito"] == arest["disciplina"]).sum()))
print("arestas com código fora de disciplinas:",
      int((~arest["pre_requisito"].isin(codigos_disc) | ~arest["disciplina"].isin(codigos_disc)).sum()))

# --- a coluna `alternativa` bate com o texto bruto? ---

bruto = disc.set_index("codigo")["prerequisitos_brutos"].fillna("")
tem_ou = arest["disciplina"].map(bruto).fillna("").str.contains(r"\bOU\b").astype(bool)
alt = arest["alternativa"].astype(bool)

print(f"\narestas com alternativa=1                         : {int(alt.sum())} de {len(arest)}")
print("alternativa=1 sem 'OU' no texto bruto (esperado 0):", int((alt & ~tem_ou).sum()))
print("alternativa=0 com 'OU' no texto bruto (esperado 0):", int((~alt & tem_ou).sum()))

# --- cada "OU" tem no máximo UMA opção dentro da grade? (as demais são códigos antigos ou de outros cursos) ---

import re
obrig = set(disc.loc[disc["natureza"] == "Obrigatória", "codigo"])
grupos_ou = [(cod, g[0] or g[1])
             for cod, txt in bruto.items()
             for g in re.findall(r"\(([^()]*\bOU\b[^()]*)\)|^([^()]*\bOU\b[^()]*)$", str(txt))]
com_duas = [(cod, g) for cod, g in grupos_ou if sum(c in obrig for c in re.findall(r"[A-Z]{3}\d{4}", g)) > 1]

print(f"\ngrupos 'OU' no texto bruto                          : {len(grupos_ou)}")
print(f"grupos 'OU' com 2+ opções dentro da grade (esperado 0): {len(com_duas)}")


---
# 3. Modelagem dos dados em um grafo

## 3.1 O que é cada elemento

| Elemento | Definição neste projeto | Por quê |
|---|---|---|
| **Nó** | disciplina **obrigatória** (identificada pelo código). Atributos: `nome`, `ch`, `natureza`, `nivel` | são as disciplinas que todo aluno precisa cursar |
| **Aresta** | `A → B` quando A é pré-requisito de B | é a relação que bloqueia: B só pode ser cursada depois de A |
| **Direção** | **sim**: da disciplina-base (A) para a que depende dela (B) | pré-requisito não é simétrico |
| **Peso** | carga horária (h) da **origem** (o pré-requisito). | uma dependência de 90 h "pesa" mais que uma de 30 h |
| **Atributo `alternativa`** | 1 quando o pré-requisito tem um "OU" com alternativa **fora da grade** (currículo antigo ou equivalente de outro curso) | é só informativo: sinaliza que existe outro caminho externo, mas não altera a estrutura da grade |

**Tipo de grafo:** dirigido, ponderado e simples (sem arestas paralelas nem laços). Como um pré-requisito não pode depender de si mesmo, o esperado é um **DAG** (grafo dirigido acíclico).

## 3.2 O que ficou de fora (e por quê)

- **Co-requisitos e equivalências:** não bloqueiam a matrícula do mesmo jeito que um pré-requisito.
- **Optativas e complementares:** o CSV não traz pré-requisitos para elas, e entrariam como nós soltos. *(Dá para incluí-las com `INCLUIR_OPTATIVAS = True`.)*
- **Alternativas "OU" fora da grade:** códigos de currículos antigos (ex.: `DCA0105`, que ainda é aceita como pré-requisito de `DCA3506` para quem a cursou antes da mudança) e disciplinas equivalentes de outros cursos. Não são nós do grafo, pois não fazem parte do currículo atual.

**Como ler as arestas com "OU":** toda aresta `A → B` do grafo é um pré-requisito **da grade atual**, válido para quem segue o currículo vigente. O `alternativa = 1` apenas indica que existe, além de A, uma opção externa à grade. Ele não torna a aresta menos real nem a dispensa na análise.

## 3.3 Decisões de modelagem (flags)

As flags abaixo são **decisões de modelagem**

In [ ]:
# ------------------------------------------------------------------
# Decisões de modelagem
# ------------------------------------------------------------------

INCLUIR_OPTATIVAS = False         # True: optativas/complementares entram como nós sem arestas
PESO_DA_ARESTA = "pre_requisito"  # "pre_requisito": ch da origem | "disciplina": ch do destino
TOP_N = 5                         # tamanho dos rankings nos gráficos

assert PESO_DA_ARESTA in ("pre_requisito", "disciplina")

## 3.4 Quais nós e arestas entram

Filtramos só as disciplinas obrigatórias e as arestas cujas duas pontas estão no grafo, e calculamos o peso de cada aresta.

In [ ]:
# --- nós e arestas que entram no grafo ---

nos_df = disc if INCLUIR_OPTATIVAS else disc[disc["natureza"] == "Obrigatória"]
nos_df = nos_df.copy()
nos_df["nivel"] = nos_df["nivel"].fillna(0).astype(int)     # Nivel 0 = sem período (optativas/complementares)
ch = nos_df.set_index("codigo")["ch"]

validas = arest["pre_requisito"].isin(ch.index) & arest["disciplina"].isin(ch.index)
print(f"Arestas descartadas por tocarem disciplina fora do grafo: {int((~validas).sum())}")

arestas_df = arest[validas].copy()
# peso da aresta = carga horária da disciplina
arestas_df["peso"] = arestas_df[PESO_DA_ARESTA].map(ch).astype(int)
print(f"Nós: {len(nos_df)} | arestas: {len(arestas_df)} | peso = ch de '{PESO_DA_ARESTA}'")

## 3.5 Construção do grafo · *Semana 2 — Modelagem e fundamentos*

`nx.DiGraph` cria um **grafo dirigido**. Cada disciplina vira um nó com atributos (`nome`, `ch`, `natureza`, `nivel`) e cada pré-requisito vira uma aresta com atributos (`weight` e `alternativa`). Isso é o conceito de **grafo com atributos** da Semana 2.

A tabela-resumo mostra as características básicas da rede:

- **Densidade** (Semana 2) = arestas / pares possíveis. Num grafo dirigido: m / (n·(n−1)). Valor baixo = rede esparsa (poucas das conexões possíveis existem).
- **Componentes fracamente conexos** (Semana 4): blocos de disciplinas ligadas entre si *se ignorarmos o sentido das setas*. Disciplinas sem nenhuma aresta formam componentes de 1 nó (**nós isolados**).
- **DAG:** se não houver ciclo, existe uma ordem topológica (ordem em que dá para cursar tudo respeitando os pré-requisitos).

In [ ]:
MINUSCULAS = {"De", "Da", "Do", "Das", "Dos", "E", "Em", "Para", "A", "À", "Ao", "Com", "Por"}
ROMANOS = {"I", "II", "III", "IV", "V"}

def nome_legivel(s):
    palavras = s.title().split()
    return " ".join(
        p.upper() if p.upper() in ROMANOS else (p.lower() if (i > 0 and p in MINUSCULAS) else p)
        for i, p in enumerate(palavras)
    )

G = nx.DiGraph(name="Pré-requisitos — Eng. da Computação UFRN")

for r in nos_df.itertuples(index=False):
    G.add_node(r.codigo, nome=nome_legivel(r.nome), ch=int(r.ch), natureza=r.natureza, nivel=int(r.nivel))
for r in arestas_df.itertuples(index=False):
    G.add_edge(r.pre_requisito, r.disciplina, weight=int(r.peso), alternativa=int(r.alternativa))

isolados = sorted(nx.isolates(G))

pesos_arestas = [d["weight"] for _, _, d in G.edges(data=True)]

resumo = pd.Series({
    "nós (disciplinas)": G.number_of_nodes(),
    "arestas (pré-requisitos)": G.number_of_edges(),
    "é um DAG (sem ciclos)?": nx.is_directed_acyclic_graph(G),
    "componentes fracamente conectados": nx.number_weakly_connected_components(G),
    "nós isolados (sem pré-requisito e sem dependentes)": len(isolados),
    "densidade": round(nx.density(G), 4),
    "carga horária somada dos nós (h)": sum(d["ch"] for _, d in G.nodes(data=True)),
    "peso das arestas, mín – máx (h)": f"{min(pesos_arestas)} – {max(pesos_arestas)}",
}, name="valor")

display(resumo.to_frame())
print("Isolados:", ", ".join(f"{v} ({G.nodes[v]['nome']})" for v in isolados) or "nenhum")

assert nx.is_directed_acyclic_graph(G), "O grafo tem ciclo: confira arestas.csv antes de seguir."

## 3.6 Visualização do grafo · *Semana 4 — visualização de redes*

- **Posição:** o eixo x é o período do currículo (`nivel`). Dentro de cada coluna, a ordem vertical vem da heurística do **baricentro**, que aproxima cada nó dos seus vizinhos para reduzir cruzamentos de setas.
- **Setas que pulam períodos:** uma aresta do 3º ao 6º período, por exemplo, ganha uma posição livre em cada coluna intermediária (um "nó virtual" invisível) e é desenhada como uma linha quebrada. Assim nenhuma seta atravessa o centro de outra disciplina, o que sugeriria um pré-requisito que não existe.
- **Cor do nó:** período (rampa azul).
- **Espessura da seta:** carga horária (peso). **Tracejada:** pré-requisito que tem alternativa fora da grade ("OU").

As funções `layout_por_periodo` e `plotar_grafo` são reaproveitadas em todas as figuras da seção 4. *(A biblioteca `nxviz` citada na Semana 4 não foi usada: as figuras foram feitas com plotly.)*

In [ ]:
# ------------------------------------------------------------------
# Estilo dos gráficos
# ------------------------------------------------------------------

SUPERFICIE, TINTA, TINTA_2, MUDO = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
GRADE, EIXO = "#e1e0d9", "#c3c2b7"
AZUL, LARANJA = "#2a78d6", "#eb6834"

# rampa de um único tom (azul), do claro ao escuro: magnitude contínua
ESCALA_SEQ = [[0.0, "#cde2fb"], [0.17, "#9ec5f4"], [0.33, "#6da7ec"], [0.5, "#3987e5"],
              [0.67, "#256abf"], [0.83, "#184f95"], [1.0, "#0d366b"]]

# rampa ordinal (períodos, k-shells)
ESCALA_ORD = [[0.0, "#86b6ef"], [0.33, "#5598e7"], [0.67, "#256abf"], [1.0, "#0d366b"]]

pio.templates["curriculo"] = go.layout.Template(layout=dict(
    font=dict(family="system-ui, -apple-system, 'Segoe UI', sans-serif", size=12, color=TINTA_2),
    paper_bgcolor=SUPERFICIE, plot_bgcolor=SUPERFICIE,
    xaxis=dict(gridcolor=GRADE, linecolor=EIXO, zerolinecolor=EIXO, ticks="", automargin=True),
    yaxis=dict(gridcolor=GRADE, linecolor=EIXO, zerolinecolor=EIXO, ticks="", automargin=True),
    title=dict(font=dict(size=15, color=TINTA), x=0.0, xanchor="left"),
    colorway=[AZUL, LARANJA],
    hoverlabel=dict(bgcolor="white", font=dict(color=TINTA)),
))

pio.templates.default = "curriculo"

print("Configuração Finalizada")

In [ ]:
def layout_por_periodo(G, varreduras=24, reinicios=40, max_por_coluna=14, altura_virtual=0.35, semente=0):
    """x = período; y = ordem que reduz cruzamentos (heurística do baricentro, com reinícios).

    Aresta que pula períodos passa por um nó virtual em cada coluna intermediária: assim ela ocupa
    uma linha livre e nenhuma seta atravessa o centro de uma disciplina.
    Devolve (pos, rotas): pos[v] = (x, y) de cada disciplina; rotas[(u, v)] = pontos da linha da aresta."""
    nivel = {v: d["nivel"] for v, d in G.nodes(data=True)}
    periodo = lambda it: it[3] if isinstance(it, tuple) else nivel[it]

    colunas = {}

    for v in sorted(G):
        colunas.setdefault(nivel[v], []).append(v)

    cadeias, prox, ant = {}, {}, {}

    for u, v in G.edges:
        cadeia = [u]

        for p in range(nivel[u] + 1, nivel[v]):            # nós virtuais nas colunas entre u e v
            virt = ("virt", u, v, p)
            colunas.setdefault(p, []).append(virt)
            cadeia.append(virt)

        cadeia.append(v)
        cadeias[(u, v)] = cadeia

        for a, b in zip(cadeia, cadeia[1:]):
            if periodo(b) == periodo(a) + 1:
                prox.setdefault(a, []).append(b)
                ant.setdefault(b, []).append(a)

    periodos = sorted(colunas)

    def ordenar(ordens_ini):
        ordens, y = {p: list(o) for p, o in ordens_ini.items()}, {}

        def atribuir(p):
            for i, it in enumerate(ordens[p]):
                y[it] = (len(ordens[p]) - 1) / 2 - i

        def cruzamentos():
            total = 0
            for p in periodos:
                seg = [(y[a], y[b]) for a in ordens[p] for b in prox.get(a, [])]
                for i in range(len(seg)):
                    for j in range(i + 1, len(seg)):
                        if (seg[i][0] - seg[j][0]) * (seg[i][1] - seg[j][1]) < 0:
                            total += 1
            return total

        for p in periodos:
            atribuir(p)
        melhor, melhor_ordens = cruzamentos(), {p: list(o) for p, o in ordens.items()}
        for s in range(varreduras):
            avancando = (s % 2 == 0)
            viz = ant if avancando else prox
            for p in (periodos if avancando else periodos[::-1]):
                def chave(it):
                    vs = viz.get(it, [])
                    return (-float(np.mean([y[u] for u in vs])) if vs else -y[it], -y[it])
                ordens[p] = sorted(ordens[p], key=chave)
                atribuir(p)
            c = cruzamentos()
            if c < melhor:
                melhor, melhor_ordens = c, {p: list(o) for p, o in ordens.items()}
        return melhor, melhor_ordens

    rng = np.random.default_rng(semente)
    melhor_total, ordens = None, None
    for r in range(reinicios):
        ini = {p: list(colunas[p]) for p in periodos}

        if r > 0:
            for p in periodos:
                ini[p] = [ini[p][i] for i in rng.permutation(len(ini[p]))]

        c, o = ordenar(ini)
        if melhor_total is None or c < melhor_total:
            melhor_total, ordens = c, o

    pos_it = {}

    for p in periodos:
        ordem = ordens[p]
        sem_virtual = not any(isinstance(it, tuple) for it in ordem)

        if sem_virtual and len(ordem) > max_por_coluna:     # coluna só de disciplinas soltas: abre em sub-colunas
            sentido = -1 if p == 0 else 1
            linhas = max_por_coluna

            for i, it in enumerate(ordem):
                sub, lin = divmod(i, max_por_coluna)
                pos_it[it] = (p + sentido * 0.28 * sub, (linhas - 1) / 2 - lin)

        else:                                               # disciplina ocupa 1 unidade; ponto de passagem, só uma fração
            alturas = [altura_virtual if isinstance(it, tuple) else 1.0 for it in ordem]
            y_atual = sum(alturas) / 2
            for it, h in zip(ordem, alturas):
                pos_it[it] = (p, y_atual - h / 2)
                y_atual -= h

    pos = {v: pos_it[v] for v in G}
    rotas = {e: [pos_it[it] for it in cadeia] for e, cadeia in cadeias.items()}
    
    return pos, rotas


def plotar_grafo(G, pos, titulo, subtitulo="", cor=None, nome_cor="", escala=ESCALA_SEQ, inteiro=False,
                 arestas_destaque=(), nos_destaque=(), tamanho_no=22, altura=None, largura=None,
                 rotulos=True, fonte_rotulo=9, posicao_rotulo=None, eixo_periodos=True,
                 rotas=None, setas=True, opacidade_arestas=0.6):
    """Desenha o DiGraph. `cor`: dict nó -> valor (rampa de um tom). Destaque em laranja.
    `rotas`: linhas quebradas das arestas (vindas de layout_por_periodo); sem elas, as arestas são retas."""
    fig = go.Figure()
    nos = list(G.nodes)
    dest_a, dest_n = set(arestas_destaque), set(nos_destaque)

    pesos = [d["weight"] for _, _, d in G.edges(data=True)]
    pmin, pmax = (min(pesos), max(pesos)) if pesos else (0, 0)

    def largura_linha(p):
        return 1.0 + 2.2 * (p - pmin) / (pmax - pmin) if pmax > pmin else 1.6

    xs, ys = [pos[v][0] for v in nos], [pos[v][1] for v in nos]

    if rotas:
        ys_todos = ys + [pt[1] for r in rotas.values() for pt in r]
    else:
        ys_todos = ys

    if altura is None:                                              # ~48 px por linha de nós
        altura = int(max(700, 48 * (max(ys_todos) - min(ys_todos) + 1) + 200))
    px_x = (largura or 1250) / (max(xs) - min(xs) + 1.5)            # pixels por unidade (aprox.), p/ a ponta da seta
    px_y = altura / (max(ys_todos) - min(ys_todos) + 2.0)

    # ---- arestas: linha (tracejada se alternativa) + ponta de seta curta ----
    mx, my, mtxt = [], [], []
    for u, v, d in G.edges(data=True):
        pts = (rotas or {}).get((u, v)) or [pos[u], pos[v]]
        (x1, y1), (x0, y0) = pts[-1], pts[-2]                       # último trecho: aponta para v
        
        marcada = (u, v) in dest_a
        esmaecida = bool(dest_a) and not marcada
        cor_a = LARANJA if marcada else MUDO
        
        w = largura_linha(d["weight"]) + (1.2 if marcada else 0)
        op = 0.25 if esmaecida else (1.0 if marcada else opacidade_arestas)
        
        fig.add_trace(go.Scatter(
            x=[p[0] for p in pts], y=[p[1] for p in pts], mode="lines", hoverinfo="skip", showlegend=False,
            opacity=op, line=dict(color=cor_a, width=w, dash="dash" if d["alternativa"] else "solid")))
        
        if setas:
            comp = float(np.hypot((x1 - x0) * px_x, (y1 - y0) * px_y))
            f = min(0.9, 22 / comp) if comp > 0 else 0.9
            
            fig.add_annotation(
                x=x1, y=y1, ax=x1 - f * (x1 - x0), ay=y1 - f * (y1 - y0),
                xref="x", yref="y", axref="x", ayref="y", text="", showarrow=True,
                arrowhead=2, arrowsize=min(2.6, max(0.8, 2.6 / w)), arrowwidth=w, arrowcolor=cor_a,
                opacity=op, standoff=int(tamanho_no / 2) + 2)
        
        if len(pts) == 2:
            mx.append((pts[0][0] + pts[1][0]) / 2)
            my.append((pts[0][1] + pts[1][1]) / 2)
        else:
            mx.append(pts[len(pts) // 2][0])
            my.append(pts[len(pts) // 2][1])
        
        mtxt.append(f"<b>{u} → {v}</b><br>peso = {d['weight']} h<br>"
                    + ("tem alternativa fora da grade (OU)" if d["alternativa"] else "sem alternativa fora da grade"))
    
    if mx:   # pontos invisíveis no meio de cada aresta
        fig.add_trace(go.Scatter(x=mx, y=my, mode="markers", marker=dict(size=14, opacity=0),
                                 hoverinfo="text", hovertext=mtxt, showlegend=False))

    # ---- legenda das arestas ----
    if G.number_of_edges():
        for nome, dash, c in (("pré-requisito sem alternativa fora da grade", "solid", MUDO),
                              ("pré-requisito com alternativa fora da grade (OU)", "dash", MUDO)):
            fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", name=nome, line=dict(color=c, width=2, dash=dash)))
        
        if dest_a:
            fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", name="caminho crítico",
                                     line=dict(color=LARANJA, width=3)))

    # ---- nós ----
    marker = dict(size=tamanho_no,
                  line=dict(width=[3 if v in dest_n else 1 for v in nos],
                            color=[LARANJA if v in dest_n else TINTA_2 for v in nos]))
    if cor is None:
        marker["color"] = AZUL
    else:
        valores = [cor[v] for v in nos]
        cmin = 0 if (min(valores) >= 0 and escala is ESCALA_SEQ) else min(valores)
        cmax = max(valores) if max(valores) > cmin else cmin + 1
        barra = dict(title=dict(text=nome_cor, side="right"), thickness=12, len=0.6, outlinewidth=0)
        
        if inteiro:
            barra.update(tickmode="linear", tick0=cmin, dtick=1 if (cmax - cmin) <= 12 else 2)
        
        marker.update(color=valores, colorscale=escala, cmin=cmin, cmax=cmax, showscale=True, colorbar=barra)

    hover = []
    for v in nos:
        d = G.nodes[v]
        periodo = f"{d['nivel']}º período" if d["nivel"] else "sem período"
        t = (f"<b>{v}</b> — {d['nome']}<br>{d['ch']} h · {periodo}<br>"
             f"pré-requisitos diretos: {G.in_degree(v)} · dependentes diretos: {G.out_degree(v)}")
        
        if cor is not None:
            t += f"<br>{nome_cor}: {cor[v]:.4g}"
        
        hover.append(t)

    fig.add_trace(go.Scatter(
        x=xs, y=ys, mode="markers+text" if rotulos else "markers", marker=marker,
        text=nos if rotulos else None,
        textposition=[posicao_rotulo[v] for v in nos] if posicao_rotulo else "bottom center",
        textfont=dict(size=fonte_rotulo, color=TINTA),
        hoverinfo="text", hovertext=hover, showlegend=False))

    # ---- layout ----
    layout = dict(title=dict(text=titulo + (f"<br><sup>{subtitulo}</sup>" if subtitulo else "")),
                  height=altura, hovermode="closest",
                  margin=dict(l=20, r=20, t=90, b=110 if eixo_periodos else 40),
                  legend=dict(orientation="h", x=0, y=-0.1, yanchor="top"))
    
    if largura:
        layout["width"] = largura
    
    if eixo_periodos:
        periodos = sorted({G.nodes[v]["nivel"] for v in nos})
        layout["xaxis"] = dict(tickmode="array", tickvals=periodos,
                               ticktext=[f"{p}º" if p else "sem período" for p in periodos],
                               title="período do currículo", range=[min(xs) - 0.7, max(xs) + 0.7],
                               zeroline=False, showgrid=False)
        layout["yaxis"] = dict(visible=False, showgrid=False, zeroline=False)
    else:
        layout["xaxis"] = dict(visible=False, showgrid=False, zeroline=False)
        layout["yaxis"] = dict(visible=False, showgrid=False, zeroline=False, scaleanchor="x")
    
    fig.update_layout(**layout)
    
    return fig


pos, rotas = layout_por_periodo(G)

fig = plotar_grafo(
    G, pos, "Rede de pré-requisitos — Engenharia da Computação",
    subtitulo="cor do nó = período · espessura da seta = carga horária (h) · tracejada = pré-requisito com alternativa fora da grade (OU)",
    cor={v: d["nivel"] for v, d in G.nodes(data=True)}, nome_cor="período", escala=ESCALA_ORD, inteiro=True,
    rotas=rotas)
fig.show()

---
# 4. Métodos utilizados para análise

## 4.1 Visão geral: o que cada método responde

Todas as medidas entram em um único DataFrame, `metricas`, com **uma linha por disciplina**. No fim ele é salvo em `dados/processed/metricas_disciplinas.csv`.

| Seção | Medida | Pergunta que responde | Semana | Função (networkx) |
|---|---|---|---|---|
| 4.3 | Grau de entrada e de saída, centralidade de grau, força | Quem bloqueia mais? Quem depende de mais? | 2 e 3 (grau) · 4 (centralidade) | `in_degree`, `out_degree`, `in/out_degree_centrality` |
| 4.4 | Betweenness | Quem é ponte entre o básico e o avançado? | 4 | `betweenness_centrality` |
| 4.5 | Closeness | Quem alcança as demais rapidamente (ou é alcançada)? | 4 | `closeness_centrality` |
| 4.6 | K-core, k-shell, coreness, degeneracy | Quais disciplinas formam o núcleo mais interligado? | 3 (núcleo/periferia) · 5 | `core_number`, `k_core`, `k_shell` |
| 4.7 | Alcance transitivo e caminho crítico | O que trava se eu reprovar aqui? Qual cadeia atrasa a formatura? | 3 (travessia) | `descendants`, `ancestors`, `topological_sort` |
| 4.8 | Consolidação | Qual o ranking geral? As medidas concordam? | — | `pandas` |
| 4.9 | O que ficou de fora | Por que não usei diâmetro, excentricidade e autovetor? | 2 e 4 | — |

## 4.2 Tabela base

A tabela `metricas` (uma linha por disciplina) começa só com os atributos. Cada seção abaixo acrescenta colunas. `barras_top` e `tabela_top` são funções auxiliares para os rankings.

In [ ]:
# tabela base
metricas = pd.DataFrame.from_dict(dict(G.nodes(data=True)), orient="index")[["nome", "natureza", "nivel", "ch"]]
metricas.index.name = "codigo"

metricas["rotulo"] = metricas.index + " · " + metricas["nome"].apply(lambda s: s if len(s) <= 30 else s[:29] + "…")


def barras_top(df, colunas, titulos, n=TOP_N, altura=None):
    """Ranking horizontal (top-n) de uma ou mais colunas. Uma cor só: as disciplinas não têm ordem natural."""
    fig = make_subplots(rows=1, cols=len(colunas), subplot_titles=titulos,
                        horizontal_spacing=0.34 if len(colunas) > 1 else 0.05)
    
    for i, col in enumerate(colunas, start=1):
        s = df[col][df[col] > 0].sort_values(ascending=False, kind="stable").head(n).iloc[::-1]
        fig.add_trace(go.Bar(
            x=s.values, y=df.loc[s.index, "rotulo"], orientation="h", showlegend=False,
            marker=dict(color=AZUL, line=dict(color=SUPERFICIE, width=2)),
            customdata=s.index.to_numpy(),
            hovertemplate="<b>%{customdata}</b><br>%{y}<br>" + col + " = %{x:.4g}<extra></extra>"), row=1, col=i)
    
    fig.update_layout(height=altura or (140 + 27 * n), bargap=0.45, margin=dict(l=10, r=10, t=90, b=40))
    
    return fig


def tabela_top(df, col, extras=(), n=TOP_N):
    return df.sort_values(col, ascending=False, kind="stable")[["nome", "nivel", col, *extras]].head(n)

print(metricas.head())

## 4.3 Grau — quem é mais usado e quem mais depende · *Semana 2 (grau) · Semana 3 (entrada/saída) · Semana 4 (centralidade de grau)*

- **Grau de saída** (`out_degree`): quantas disciplinas dependem **diretamente** de A. Valor alto = **gargalo imediato**.
- **Grau de entrada** (`in_degree`): quantos pré-requisitos diretos B exige. Valor alto = disciplina que **depende de muitas** (ponto de convergência).
- **Centralidade de grau** = grau / (N − 1) (`in_degree_centrality`, `out_degree_centrality`): o mesmo número normalizado pelo tamanho da rede, entre 0 e 1.
- **Força** (`G.in_degree(weight="weight")`): soma dos pesos das arestas. Com o peso padrão (ch da origem), a força de entrada é a carga horária somada dos pré-requisitos diretos.

**Limite:** o grau só enxerga os vizinhos imediatos. Não diz nada sobre dependentes dos dependentes (isso é o alcance, na seção 4.7).

In [ ]:
metricas["grau_entrada"]        = pd.Series(dict(G.in_degree()))
metricas["grau_saida"]          = pd.Series(dict(G.out_degree()))
metricas["cent_grau_entrada"]   = pd.Series(nx.in_degree_centrality(G))
metricas["cent_grau_saida"]     = pd.Series(nx.out_degree_centrality(G))
metricas["forca_entrada_h"]     = pd.Series(dict(G.in_degree(weight="weight")))
metricas["forca_saida_h"]       = pd.Series(dict(G.out_degree(weight="weight")))

print("Maior grau de saída (mais dependentes diretos):")
display(tabela_top(metricas, "grau_saida", ["grau_entrada", "forca_saida_h", "cent_grau_saida"]))

print("Maior grau de entrada (mais pré-requisitos diretos):")
display(tabela_top(metricas, "grau_entrada", ["grau_saida", "forca_entrada_h", "cent_grau_entrada"]))

In [ ]:
# ----------------------------------------------
# Gráfico de barras com métricas
# ----------------------------------------------

fig = barras_top(metricas, 
                 ["grau_saida", "grau_entrada"],
                 ["Grau de saída: nº de dependentes diretos", "Grau de entrada: nº de pré-requisitos diretos"])
fig.update_layout(title="Grau: quem bloqueia mais e quem depende de mais")
fig.show()

In [ ]:
# ----------------------------------------------
# Grafos de entrada e saída
# ----------------------------------------------

# --- Gráfo de saída ---
fig = plotar_grafo(G, pos, "Grau de saída", subtitulo="quanto mais escuro, mais disciplinas dependem diretamente da disciplina",
                   cor=metricas["grau_saida"].to_dict(), nome_cor="grau de saída", inteiro=True, rotas=rotas)
fig.show()

# --- Grafo de entrada ---
fig = plotar_grafo(G, pos, "Grau de entrada", subtitulo="quanto mais escuro, mais dependências a disciplina tem",
                   cor=metricas["grau_entrada"].to_dict(), nome_cor="grau de entrada", inteiro=True, rotas=rotas)
fig.show()

## 4.4 Betweenness — quem está no meio dos caminhos · *Semana 4*

Para cada disciplina v, é a fração dos caminhos mínimos entre pares (s, t) que passam por v. Num grafo dirigido, os caminhos seguem o sentido das setas, então só contam pares em que t pode ser alcançado a partir de s. O valor é normalizado por (N − 1)(N − 2).

- Disciplinas no **início** (sem pré-requisito) ou no **fim** (sem dependentes) de uma cadeia têm betweenness 0: nenhum caminho passa *por* elas.
- Duas versões: `betweenness` (distância = número de arestas) e `betweenness_ponderada_h` (distância = soma das horas dos pesos).

In [ ]:
metricas["betweenness"]             = pd.Series(nx.betweenness_centrality(G, normalized=True))
metricas["betweenness_ponderada_h"] = pd.Series(nx.betweenness_centrality(G, normalized=True, weight="weight"))

zeros = int((metricas["betweenness"] == 0).sum())

print(f"Disciplinas com betweenness = 0 (início/fim de cadeia ou isoladas): {zeros} de {len(metricas)}")
display(tabela_top(metricas, "betweenness", ["betweenness_ponderada_h", "grau_entrada", "grau_saida"]))

fig = barras_top(metricas, ["betweenness", "betweenness_ponderada_h"],
                 ["Betweenness (caminhos por nº de arestas)", "Betweenness ponderada (caminhos por horas)"])
fig.update_layout(title="Betweenness: as disciplinas-ponte")
fig.show()

fig = plotar_grafo(G, pos, "Betweenness", subtitulo="quanto mais escuro, mais caminhos entre disciplinas passam por ela",
                   cor=metricas["betweenness"].to_dict(), nome_cor="betweenness", rotas=rotas)
fig.show()

## 4.5 Closeness — quão perto está das demais · *Semana 4*

closeness(v) = (N − 1) / (soma das distâncias de v aos demais).

Num DAG muitos pares **não têm caminho** (não há como ir de uma disciplina final a uma inicial). O networkx aplica a correção de **Wasserman–Faust**, que multiplica o valor pela fração de nós alcançáveis, e nós isolados ficam com 0. São duas versões, uma por sentido:

- `closeness_saida` (grafo invertido, distâncias **a partir de** v): alto quando v alcança muitas disciplinas rapidamente. Tende a destacar a **base** do currículo.
- `closeness_entrada` (padrão do networkx, distâncias **até** v): alto quando muitas disciplinas chegam a v rapidamente. Tende a destacar disciplinas **avançadas**.

In [ ]:
metricas["closeness_entrada"] = pd.Series(nx.closeness_centrality(G))
metricas["closeness_saida"] = pd.Series(nx.closeness_centrality(G.reverse(copy=False)))

print("Maior closeness de saída (alcançam muitas disciplinas rapidamente):")
display(tabela_top(metricas, "closeness_saida", ["closeness_entrada", "grau_saida"]))

print("Maior closeness de entrada (alcançadas rapidamente por muitas):")
display(tabela_top(metricas, "closeness_entrada", ["closeness_saida", "grau_entrada"]))

fig = barras_top(metricas, ["closeness_saida", "closeness_entrada"],
                 ["Closeness de saída (alcança os demais)", "Closeness de entrada (é alcançada)"])
fig.update_layout(title="Closeness nos dois sentidos")
fig.show()

## 4.6 K-core, k-shell e coreness (decomposição em núcleos) · *Semana 5 (e Semana 3: núcleo × periferia)*

- **k-core**: o maior subgrafo em que todo vértice tem grau ≥ k *dentro do próprio subgrafo*. Obtém-se removendo repetidamente os vértices com grau < k (`nx.k_core`).
- **Coreness** (core number) de v: o maior k tal que v pertence ao k-core (`nx.core_number`).
- **k-shell**: vértices com coreness exatamente k, ou seja, k-core menos (k+1)-core (`nx.k_shell`). Os k-cores se **aninham** como camadas de uma cebola; os k-shells **particionam** o grafo.
- **Degeneracy**: o maior k cujo k-core não é vazio.

O k-core ignora a direção das arestas. Usamos `G.to_undirected()`, o que equivale ao grau total (entrada + saída); como o DAG não tem arestas recíprocas, nada é contado duas vezes.

**Resumo:** shells externas = **periferia**. O core interno reúne as disciplinas mais densamente interligadas. A coreness mede densidade local de conexões, **não** o efeito cascata.

Duas figuras: um **controle deslizante** que mostra o k-core para cada k, e a **visão cebola**, com a k-shell mais alta no centro.

In [ ]:
U = G.to_undirected()
coreness = nx.core_number(U)
metricas["coreness"] = pd.Series(coreness)
k_max = max(coreness.values())

cores = {k: nx.k_core(U, k, core_number=coreness) for k in range(k_max + 1)}
shells = {k: nx.k_shell(U, k, core_number=coreness) for k in range(k_max + 1)}

resumo_cores = pd.DataFrame({
    "|k-core| (acumulado)": [cores[k].number_of_nodes() for k in range(k_max + 1)],
    "|k-shell| (partição)": [shells[k].number_of_nodes() for k in range(k_max + 1)],
}, index=pd.Index(range(k_max + 1), name="k"))

resumo_cores["% do grafo no k-core"] = (100 * resumo_cores["|k-core| (acumulado)"] / G.number_of_nodes()).round(1)

display(resumo_cores)

print(f"Degenerescência (maior k com k-core não vazio): {k_max}\n")
for k in range(k_max, -1, -1):
    membros = sorted(shells[k].nodes)
    print(f"{k}-shell ({len(membros)}): {', '.join(membros)}")

In [ ]:
# ----------------------------------------------
# Grafo completo interativo com k-core
# ----------------------------------------------

MODO_K = "core"   # "core": mostra coreness ≥ k (k-core, aninhado) · "shell": só coreness = k (k-shell)
grupos = cores if MODO_K == "core" else shells

# uma figura por k
figs = {k: plotar_grafo(G.subgraph(grupos[k].nodes), pos, "", cor=coreness, nome_cor="coreness",
                        escala=ESCALA_ORD, inteiro=True, rotas=rotas)
        for k in range(k_max + 1)}

def titulo_k(k):
    n = grupos[k].number_of_nodes()
    filtro = f"coreness ≥ {k}" if MODO_K == "core" else f"coreness = {k}"
    return (f"{k}-{MODO_K}: {n} de {G.number_of_nodes()} disciplinas"
            f"<br><sup>{filtro} · arraste o controle para trocar o k · cor = coreness</sup>")

# junta os traços de todas as figuras numa só
fig = go.Figure(layout=figs[0].layout)          # k = 0 é o grafo inteiro: eixos e altura de referência
faixa, inicio = {}, 0

for k, f in figs.items():
    fig.add_traces(list(f.data))
    faixa[k] = range(inicio, inicio + len(f.data))
    inicio += len(f.data)

for i, tr in enumerate(fig.data):
    tr.visible = i in faixa[0]

fig.update_traces(selector=dict(mode="markers+text"), marker=dict(cmin=0, cmax=k_max))   # mesma escala de cor em todo k

passos = [dict(method="update", label=str(k),
               args=[{"visible": [i in faixa[k] for i in range(inicio)]},
                     {"annotations": [a.to_plotly_json() for a in figs[k].layout.annotations],   # setas
                      "title.text": titulo_k(k)}])
          for k in figs]

fig.update_layout(title_text=titulo_k(0), margin_b=230,
                  sliders=[dict(active=0, steps=passos, y=-0.16, yanchor="top", len=0.9, x=0.05,
                                currentvalue=dict(prefix="k = ", font=dict(size=14)))])

fig.show()

In [ ]:
# ----------------------------------------------
# Visão cebola: k-shell mais alta no centro, coreness 0 na borda
# ----------------------------------------------

def layout_cebola(shells_por_k, espaco=0.5, folga=0.85):
    pos_c, raio, primeiro = {}, 0.0, True

    for k in sorted(shells_por_k, reverse=True):
        nos = sorted(shells_por_k[k].nodes)
        
        if not nos:
            continue
        
        minimo = 0.0 if (primeiro and len(nos) == 1) else espaco * len(nos) / (2 * np.pi)
        raio = minimo if primeiro else max(minimo, raio + folga)
        
        for i, v in enumerate(nos):
            ang = 2 * np.pi * i / len(nos) + (np.pi / 2 if len(nos) > 1 else 0)
            pos_c[v] = (raio * np.cos(ang), raio * np.sin(ang))
        
        primeiro = False
    
    return pos_c


def posicao_radial(pos_c):
    """Rótulo para fora do círculo, conforme o ângulo do nó."""
    setores = ["middle right", "top right", "top center", "top left",
               "middle left", "bottom left", "bottom center", "bottom right"]
    out = {}
    
    for v, (x, y) in pos_c.items():
        if abs(x) < 1e-9 and abs(y) < 1e-9:
            out[v] = "bottom center"
        else:
            ang = np.degrees(np.arctan2(y, x)) % 360
            out[v] = setores[int(((ang + 22.5) % 360) // 45)]
    
    return out


pos_cebola = layout_cebola(shells)
fig = plotar_grafo(G, pos_cebola, "K-core decomposition em camadas",
                   subtitulo="centro = k-shell mais alta · borda = coreness 0 · cor = coreness",
                   cor=coreness, nome_cor="coreness", escala=ESCALA_ORD, inteiro=True,
                   tamanho_no=16, altura=900, largura=900, fonte_rotulo=8,
                   posicao_rotulo=posicao_radial(pos_cebola), eixo_periodos=False,
                   setas=False, opacidade_arestas=0.2)     # k-core ignora a direção: arestas só como fundo
fig.show()

## 4.7 Alcance transitivo e caminho crítico · *Semana 3 (travessia/alcançabilidade)

Num DAG de pré-requisitos, o efeito cascata é dado pelos **descendentes** de A (tudo que se alcança seguindo as setas a partir dela). Esse é o conceito de **alcançabilidade por travessia** da Semana 3 (o networkx usa busca em largura por baixo). É a medida mais direta para "ponto crítico".

- `alcance_descendentes`: `len(nx.descendants(G, v))`. `horas_bloqueadas_h`: carga horária somada desses descendentes.
- `alcance_ancestrais`: `len(nx.ancestors(G, v))`. É tudo que precisa ter sido cursado antes de v.
- **Caminho crítico**: a cadeia mais longa de pré-requisitos, que impõe um mínimo de períodos para se formar *se só os pré-requisitos contassem*.

In [ ]:
ordem_topologica = list(nx.topological_sort(G))

desc = {v: nx.descendants(G, v) for v in G}
metricas["alcance_descendentes"] = pd.Series({v: len(s) for v, s in desc.items()})
metricas["horas_bloqueadas_h"] = pd.Series({v: sum(G.nodes[d]["ch"] for d in s) for v, s in desc.items()})
metricas["alcance_ancestrais"] = pd.Series({v: len(nx.ancestors(G, v)) for v in G})

# maior cadeia que chega em v (antes) e que parte de v (depois), em nº de arestas
antes = {v: 0 for v in G}
for v in ordem_topologica:
    for s in G.successors(v):
        antes[s] = max(antes[s], antes[v] + 1)
depois = {v: 0 for v in G}

for v in reversed(ordem_topologica):
    for p in G.predecessors(v):
        depois[p] = max(depois[p], depois[v] + 1)
maior_cadeia = max(antes[v] + depois[v] for v in G)


def caminho_mais_pesado(valor_no):
    """Cadeia de pré-requisitos com maior soma de `valor_no` (programação dinâmica no DAG)."""
    acum, ant = {}, {}
    for v in ordem_topologica:
        p = max(G.predecessors(v), key=acum.get, default=None)
        acum[v] = valor_no(v) + (acum[p] if p is not None else 0)
        ant[v] = p
    v = max(acum, key=acum.get)
    caminho = []
    while v is not None:
        caminho.append(v)
        v = ant[v]
    return caminho[::-1], max(acum.values())


caminho_h, total_h = caminho_mais_pesado(lambda v: G.nodes[v]["ch"])
caminho_n, _ = caminho_mais_pesado(lambda v: 1)
total_n_h = sum(G.nodes[v]["ch"] for v in caminho_n)
print(f"Caminho crítico em horas   : {total_h} h, {len(caminho_h)} disciplinas\n  " + " → ".join(caminho_h))
print(f"Cadeia mais longa em nº de disciplinas: {len(caminho_n)} disciplinas, {total_n_h} h\n  " + " → ".join(caminho_n))

display(tabela_top(metricas, "alcance_descendentes",
                   ["horas_bloqueadas_h", "alcance_ancestrais", "grau_saida"]))

fig = barras_top(metricas, ["alcance_descendentes", "horas_bloqueadas_h"],
                 ["Alcance: nº de disciplinas bloqueadas", "Carga horária bloqueada (h)"])
fig.update_layout(title="Efeito cascata: o que trava se a disciplina não for cursada")
fig.show()

fig = plotar_grafo(G, pos, "Alcance transitivo e caminho crítico",
                   subtitulo="cor = nº de disciplinas bloqueadas (descendentes) · laranja = caminho crítico em horas",
                   cor=metricas["alcance_descendentes"].to_dict(), nome_cor="descendentes",
                   arestas_destaque=set(zip(caminho_h, caminho_h[1:])), nos_destaque=set(caminho_h), rotas=rotas)
fig.show()

## 4.8 Consolidação — ranking geral e concordância entre as medidas

- **Índice de criticidade** (heurística simples): média dos *percentis* de `grau_saida`, `betweenness`, `alcance_descendentes` e `coreness`.
- **Correlação de Spearman** entre as medidas (correlação entre *rankings*, de −1 a 1): mostra onde as medidas concordam ou discordam. Por exemplo, "ser ponte" (betweenness) e "estar no núcleo denso" (coreness) podem apontar disciplinas diferentes, e discutir essa diferença é o que responde bem à pergunta "o que é ponto crítico".

No fim, a tabela completa é salva em CSV.

In [ ]:
COLUNAS_INDICE = ["grau_saida", "betweenness", "alcance_descendentes", "coreness"]
metricas["indice_criticidade"] = metricas[COLUNAS_INDICE].rank(pct=True).mean(axis=1).round(3)

display(metricas.sort_values("indice_criticidade", ascending=False, kind="stable")
        [["nome", "nivel", "ch", *COLUNAS_INDICE, "indice_criticidade"]].head(TOP_N))

fig = barras_top(metricas, ["indice_criticidade"], ["Índice de criticidade (média dos percentis)"], altura=520)
fig.update_layout(title="Ranking geral de criticidade")
fig.show()

# correlação de Spearman entre as medidas
cols_corr = ["grau_entrada", "grau_saida", "betweenness", "closeness_entrada", "closeness_saida",
             "coreness", "alcance_descendentes", "horas_bloqueadas_h", "alcance_ancestrais"]

corr = metricas[cols_corr].corr(method="spearman")

fig = go.Figure(go.Heatmap(
    z=corr.values, x=cols_corr, y=cols_corr, zmin=-1, zmax=1, zmid=0,
    colorscale=[[0.0, "#e34948"], [0.5, "#f0efec"], [1.0, "#2a78d6"]],
    text=np.round(corr.values, 2), texttemplate="%{text}", textfont=dict(size=11),
    hovertemplate="%{y} × %{x}<br>Spearman = %{z:.3f}<extra></extra>",
    colorbar=dict(title=dict(text="Spearman", side="right"), thickness=12, outlinewidth=0)))

fig.update_layout(title="Correlação de Spearman entre as medidas", height=650,
                  yaxis=dict(autorange="reversed"), xaxis=dict(tickangle=-40))

fig.show()

tabela_final = metricas.drop(columns="rotulo")
tabela_final.round(5).to_csv(ARQ_SAIDA, encoding="utf-8")

print(f"Tabela completa salva em '{ARQ_SAIDA}' ({tabela_final.shape[0]} linhas x {tabela_final.shape[1]} colunas)")
# display(tabela_final.sort_values("indice_criticidade", ascending=False, kind="stable").round(4))

## 4.9 O que ficou de fora, e por quê

Medidas do curso que **não** foram calculadas, com a justificativa:

- **Excentricidade, diâmetro, raio, centro e periferia** (Semana 4): dependem de distâncias entre *todos* os pares e exigem grafo (fortemente) conexo. Um DAG nunca é fortemente conexo, porque não há caminho de volta, e este grafo ainda tem componentes separados. Só seriam calculáveis ignorando a direção **e** restringindo a análise ao maior componente, o que apagaria o sentido de "pré-requisito" e deixaria disciplinas soltas de fora.
- **Centralidade de autovetor** (Semana 4): num DAG a **matriz de adjacência** (Semana 2) é *nilpotente* (A^N = 0, todos os autovalores são zero), então o método da potência não converge e a medida é degenerada.
- **Triângulos e clustering** (Semana 4), **k-truss** e **Gephi** (Semana 5), **bipartidas e projeção** (Semana 6): não respondem à pergunta do projeto.

A célula abaixo mostra isso na prática com o próprio grafo.

In [ ]:
print("Fortemente conexo?", nx.is_strongly_connected(G))
print("Componentes fracamente conexos:", nx.number_weakly_connected_components(G))

for nome, chamada in (("nx.diameter(G)", lambda: nx.diameter(G)),
                      ("nx.eigenvector_centrality(G)", lambda: nx.eigenvector_centrality(G, max_iter=200))):
    try:
        chamada()
        print(f"{nome} -> calculou sem erro")
    except nx.NetworkXException as e:
        print(f"{nome} -> {type(e).__name__}: {str(e)[:90]}")

print("Componentes fortemente conexos:", nx.number_strongly_connected_components(G),
      "(num DAG, cada disciplina é o seu próprio SCC)")

print("Maior componente fracamente conexo (GCC):", len(max(nx.weakly_connected_components(G), key=len)), "disciplinas")

# matriz de adjacência: num DAG, A^N = 0 (nilpotente) -> autovalores todos zero
A = nx.to_numpy_array(G, weight=None)
print("A^N é a matriz nula (N = nº de nós)?", not np.linalg.matrix_power(A, len(G)).any())

---
# 5. Como essa análise é interpretada

## 5.1 Como ler cada medida

| Medida | Valor **alto** significa… | Cuidado ao interpretar |
|---|---|---|
| **Grau de saída** | muitas disciplinas dependem **diretamente** dela (gargalo imediato) | ignora dependentes indiretos |
| **Grau de entrada** | ela exige muitos pré-requisitos (ponto de convergência) | está no *fim* das cadeias: não bloqueia outras, é "vítima" dos atrasos |
| **Alcance (descendentes) e horas bloqueadas** | muitas disciplinas ficam travadas, direta ou indiretamente (**efeito cascata**) | é a medida mais direta de "crítico" para quem planeja o curso |
| **Betweenness** | muitos caminhos mínimos passam por ela (**ponte**) | valores absolutos pequenos; compare **posições no ranking**. Início e fim de cadeia sempre dão 0 |
| **Closeness de saída** | alcança muitas disciplinas em poucos saltos (**base** do curso) | vale para disciplinas do começo |
| **Closeness de entrada** | muitas disciplinas chegam a ela em poucos saltos (**reta final**) | vale para disciplinas do fim (TCC, estágio) |
| **Coreness / k-shell** | está no miolo **densamente interligado** | **não** mede efeito cascata. Uma disciplina de base pode ter coreness baixa e bloquear metade do curso |
| **Índice de criticidade** | é bem ranqueada em vários critérios ao mesmo tempo | é uma heurística de ordenação, não uma teoria |

## 5.2 As medidas respondem a perguntas diferentes

O mais importante da interpretação é que **"crítico" não é uma coisa só**. A correlação de Spearman (seção 4.9) mostra isso:

- Grau de saída, closeness de saída e alcance andam **juntos** (correlações altas, 0,7 a 0,9): quem tem muitos dependentes diretos costuma também travar muita coisa no total.
- **Coreness quase não se relaciona com alcance** (≈ 0,1): estar no núcleo denso não significa bloquear mais. São dois tipos de importância distintos.
- Medidas de entrada (grau de entrada, closeness de entrada, ancestrais) têm correlação **negativa** com as de saída/alcance: o currículo tem um lado "base" (bloqueia) e um lado "fim" (depende). Disciplinas do começo e do fim são críticas por motivos opostos.
- Betweenness tem correlação **moderada** (0,4 a 0,5) com quase todas: é um terceiro tipo de importância, a de quem está no meio do caminho.

## 5.3 Cuidados metodológicos

1. **Alternativas "OU" fora da grade.** A rede descreve o caminho de quem segue a grade atual. Um aluno que cursou um equivalente antigo ou de outro curso pode não depender daquela aresta, então para ele o efeito cascata real é menor. Isso não muda a estrutura da grade: só indica que a criticidade medida vale para o caminho padrão.
2. **A rede mede estrutura, não risco real.** Não há dados de reprovação por disciplina. "Crítica" aqui quer dizer *estruturalmente importante*, não *difícil* nem *com muitas reprovações*.
3. **O peso quase não muda o ranking.** Como o peso de cada aresta é a carga horária da origem (entre 30 e 120 h), as versões ponderadas ficam muito parecidas com as não ponderadas.
4. **Rede pequena (50 nós):** pequenas mudanças de modelagem podem trocar posições vizinhas no ranking. Por isso olhamos grupos de disciplinas e vários critérios, e não um único número.

---
# 6. Resultados

## 6.1 A rede em números

- **50 disciplinas obrigatórias** e **87 pré-requisitos** (46 deles com uma alternativa "OU" fora da grade), carga horária total de **3.014 h**.
- É um **DAG** (sem ciclos). Densidade **0,0355**: rede esparsa.
- **4 componentes fracamente conexos**, sendo **3 disciplinas isoladas** (ECT3107, ECT3203, ECT3305), sem pré-requisito e sem dependentes. O resto forma um único bloco grande.
- Maior cadeia de pré-requisitos: **7 disciplinas** (em 10 períodos).

## 6.2 Resposta à pergunta

Os pontos críticos aparecem em **quatro grupos**, cada um por um motivo diferente:

**1. A fundação que trava o curso inteiro (efeito cascata)**

- `ECT3101` Matemática Básica e Modelagem
- `ECT3104` Lógica de Programação
- `ECT3201` Linguagem de Programação
- `ECT3207` Cálculo I.
  
Cada uma bloqueia, direta ou indiretamente, entre **21 e 24 das outras 49 disciplinas** (quase metade do curso e de **1.499 a 1.679 h**). `ECT3101` lidera: 24 disciplinas e 1.679 h bloqueadas. São de 1º e 2º períodos, então um atraso ali se propaga por todo o curso.
**Atenção:** o índice de criticidade subestima `ECT3101` (0,590) e `ECT3104` (0,410), porque são início de cadeia (betweenness 0) e têm coreness baixa. Por isso a pergunta não deve ser respondida só com o índice: o alcance mostra o que o índice esconde.

**2. Os hubs e pontes do meio do curso**
`DCA3303` Programação Avançada tem o maior grau de saída (**9 dependentes diretos**) e está entre as três maiores betweenness. `ECT3201` (5 dependentes diretos) e `ECT3207` (4) completam o trio de maior grau de saída e aparecem entre as cinco maiores em betweenness e closeness de saída. Elas ligam o básico ao avançado. No índice de criticidade, ficam em primeiro (`ECT3201` e `DCA3303`, 0,908) e terceiro (`ECT3207`, 0,892).

**3. Os pontos de convergência do fim do curso**
`DCA3801` Projeto Integrado (**12 pré-requisitos diretos**) tem a maior betweenness da rede, junto de `ECT3517`, que liga o miolo à reta final. `ECP3902` Estágio (12) e `ECP3901` TCC (7) têm os maiores graus de entrada. Elas não travam outras: são as que **mais sofrem** com atrasos anteriores.

**4. O núcleo denso**
A decomposição em k-cores dá **degeneracy 3**. O 3-core tem **26 disciplinas (52 %)**, majoritariamente disciplinas `DCA` mais `ECT3201`, `ECT3202`, `ECT3207`, `ECT3302`, `ECT3304`, `ECT3401` e `ECT3511`. Há ainda 12 disciplinas na 2-shell, 9 na 1-shell e 3 na 0-shell (as isoladas). **Nem todas as mais bloqueadoras estão no núcleo:** `ECT3101` está na 2-shell e `ECT3104` na 1-shell, apesar de bloquearem mais que a maioria. Coreness e efeito cascata medem coisas diferentes.

**Caminho crítico (planejamento).** A cadeia mais pesada tem **619 h** e 7 disciplinas:

`ECT3101 → ECT3207 → ECT3302 → ECT3511 → DCA3506 → DCA3801 → ECP3901`

## 6.3 Tabela-resposta

In [ ]:
# Tabela-resposta: as disciplinas que aparecem como críticas em algum critério
criticas = ["ECT3101", "ECT3104", "ECT3201", "ECT3207", "DCA3303", "ECT3517", "DCA3801"]
colunas_resp = ["nome", "nivel", "grau_saida", "grau_entrada", "alcance_descendentes", "horas_bloqueadas_h",
                "betweenness", "closeness_saida", "coreness", "indice_criticidade"]
display(metricas.loc[criticas, colunas_resp].round(4))

print("Caminho crítico em horas:", " → ".join(caminho_h), f"({total_h} h)")

## 6.4 Limitações

- A rede descreve a **estrutura do currículo**, não o comportamento dos alunos (taxas de reprovação, oferta de turmas, trancamentos).
- As alternativas "OU" fora da grade (códigos antigos e equivalentes de outros cursos) não entram na rede; a marca `alternativa` apenas as sinaliza. Alunos com esses equivalentes podem ter um efeito cascata menor.
- Co-requisitos, equivalências, optativas e complementares ficaram de fora.
- O peso é a carga horária, que varia pouco (30–120 h) e muda pouco os resultados.
- 50 nós é uma rede pequena: o ranking entre posições vizinhas é frágil.

## 6.5 Conteúdo do curso utilizado (para o README)

Formato pedido: **conceito · semana · onde aparece · para que serviu**. "Onde aparece" usa as seções deste notebook.

| Conceito | Semana | Onde aparece | Para que serviu |
|---|---|---|---|
| Grafo dirigido, ponderado e DAG (`nx.DiGraph`) | 2 | 3.1 e 3.5 | modelar o currículo: nó = disciplina, aresta = pré-requisito, peso = carga horária |
| Atributos de nós e arestas | 2 | 3.5 | guardar nome, carga horária, período, peso e a marca de alternativa fora da grade |
| Densidade (`nx.density`) | 2 | 3.5 | medir o quão esparsa é a rede |
| Subredes (`G.subgraph`, `nx.k_core`) | 2 | 4.6 | analisar só o núcleo e comparar com o resto |
| Grau, grau de entrada e de saída | 2 e 3 | 4.3 (`in_degree`, `out_degree`) | achar gargalos imediatos e disciplinas dependentes |
| Matriz de adjacência (nilpotente em DAG) | 2 | 4.10 | justificar por que o autovetor foi descartado |
| Alcançabilidade por travessia (BFS por baixo de `descendants`/`ancestors`) | 3 | 4.7 | medir efeito cascata: quantas disciplinas ficam bloqueadas |
| Núcleo e periferia | 3 | 4.6 | separar o miolo interligado das pontas |
| Caminhos e distâncias | 4 | 4.4 e 4.5 | base de betweenness e closeness |
| Centralidade de grau (normalizada) | 4 | 4.3 (`*_degree_centrality`) | comparar graus em escala 0–1 |
| Betweenness | 4 | 4.4 (`betweenness_centrality`) | achar disciplinas-ponte entre o básico e o avançado |
| Closeness (com correção de Wasserman–Faust) | 4 | 4.5 (`closeness_centrality`) | medir quem alcança (ou é alcançada por) muitas disciplinas rapidamente |
| Componentes fraco/forte e GCC | 4 | 3.5 e 4.10 | identificar blocos separados e isolados; mostrar que o DAG não é fortemente conexo |
| Visualização de redes | 4 | 3.6 (`plotar_grafo`) | desenhar o currículo por período, com cor e espessura |
| K-core, k-shell, core number, degeneracy | 5 | 4.6 (`core_number`, `k_core`, `k_shell`) | separar o núcleo densamente interligado da periferia |
| Camadas de cebola (layout concêntrico por k-shell) | 5 | 4.6 (`layout_cebola`) | visualizar núcleo × periferia |
| Excentricidade, diâmetro, periferia, centro, autovetor | 2 e 4 | 4.10 | **avaliados e descartados** (justificativa no notebook) |
| Caminho crítico | método próprio | 4.7 | medir a cadeia que atrasa a formatura |
| Índice de criticidade (média de percentis) e Spearman | método próprio | 4.9 | consolidar critérios e ver onde as medidas discordam |
| *Não usados:* BFS/DFS manuais, snowball, triângulos/clustering, k-truss, bipartidas/projeção/Jaccard/NER | 3, 4, 5, 6 | — | não se aplicam à pergunta |